<h1>
    This file generates g-t diagram of each unit with model prediction & declare line
</h1>

In [1]:
import os
import sys

import plotly.graph_objects as go

current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)

from src.models.filter_data.filter_data import *
from src.models.filter_data.feature_adder import *
from joblib import load
import plotly.express as px


In [2]:
csv_read_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
df = pd.read_csv(csv_read_path, encoding='utf-8')

In [3]:
csv_read_path = os.path.join(project_root, "data", "interim", "factors.csv")
df_factors = pd.read_csv(csv_read_path)
coefs = get_coefs(df_factors)

In [4]:
color_map = {
    0: "purple",
    1: "orange",
    2: "pink",
    3: "blue",
    4: "black",
    5: "red",
    6: "green"
}

<h2>
    Draw generation vs temperature plot
</h2>

In [5]:
def add_model_prediction(fig, df, x_col, name, code, project_root, model_subdir, trace_name, dash="solid", width=2):
    try:
        folder_path = os.path.join(project_root, "src", "models", "fitted_models")
        model_path = f"{folder_path}/{model_subdir}/{name}_{code}.joblib"
        model = load(model_path)

        x_line = np.linspace(df[x_col].min(), df[x_col].max(), 100)
        df_line = pd.DataFrame(x_line, columns=[x_col])
        y_line = model.predict(df_line)

        fig.add_trace(
            go.Scatter(x=df_line[x_col], y=y_line, mode="lines", name=trace_name, line=dict(dash=dash, width=width),
                       visible="legendonly"))
    except:
        pass



In [6]:
def draw_gen_temp_plot(df, x_col, y_col, coefs, name, code, color_map, save=False, project_root=None):
    fig = go.Figure()

    for peak_value, color in color_map.items():
        df_subset = df[df["is_good_peak"] == peak_value]
        fig.add_trace(go.Scatter(
            x=df_subset[x_col],
            y=df_subset[y_col],
            mode="markers",
            marker=dict(size=4, color=color),
            name=f"is_good_peak = {peak_value}",
            hovertext=df_subset["datetime"]
        ))

    fig.update_traces(marker=dict(size=4, sizemode="diameter", sizeref=1, opacity=0.7))

    a, b = coefs.get((name, code))
    x_line = np.linspace(df[x_col].min(), df[x_col].max(), 100)
    y_line = a * x_line + b

    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        name=f"y = {a:.3f}x + {b:.3f}",
        line=dict(dash="dash", width=2)
    ))

    add_model_prediction(fig, df, x_col, name, code, project_root, "normal", "predict")
    add_model_prediction(fig, df, x_col, name, code, project_root, "turbo", "turbo predict")

    if save and project_root:
        save_path = f"{project_root}/src/visualization/unit_figs/test_models/{name}-{code}_temp.html"
        fig.write_html(save_path)
    else:
        fig.show()


In [10]:
power_plants = df[['name', 'code']].drop_duplicates()

for row in power_plants.itertuples():
    name_plot, code_plot = row.name, row.code
    ds_n_c_plot = Data_selector(Data_selector(df).select_peaks(goodness=2, is_tight=False))
    df_n_c_plot = ds_n_c_plot.filter_name_code(name_plot, code_plot)
    try:
        draw_gen_temp_plot(df_n_c_plot, "temperature", "generation", coefs, name_plot, code_plot, color_map, save=True,
                           project_root=project_root)
    except Exception as e:
        print(f"This error occurred while drawing diagram related to {name_plot}-{code_plot}:\n {e}")

<h2>
    Draw generation vs date plot
</h2>

In [8]:
def draw_gen_date_plot(df_m1, name, code, save=False):

    fig = px.scatter(
        df_m1,
        x="datetime",
        y='generation',
        color='is_good_peak',
        title='Generation over Time',
        labels={'generation': 'Generation', 'datetime': 'Time'},
        hover_data=['datetime', 'generation', "temperature"]
    )

    if save:
        path = f"{project_root}/src/visualization/unit_figs/test_models/{name}-{code}_date.html"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        fig.write_html(path)
    else:
        fig.show()

In [9]:
power_plants = df[['name', 'code']].drop_duplicates()

for row in power_plants.itertuples():
    name_plot, code_plot = row.name, row.code
    ds_n_c_plot = Data_selector(Data_selector(df).select_peaks(goodness=2, is_tight=False))
    df_n_c_plot = ds_n_c_plot.filter_name_code(name_plot, code_plot)
    try:
        draw_gen_date_plot(df_n_c_plot, name_plot, code_plot, save=True)
    except Exception as e:
        print(f"This error occurred while drawing diagram related to {name_plot}-{code_plot}:\n {e}")